# 🎬 Chrome Extension → Colab 무료 영상 편집 파이프라인 v2.0\n\n**Chrome Extension에서 수집한 이미지를 Colab에서 무료로 영상 편집**\n\n## 📌 주요 기능:\n- JSON 파일 직접 업로드 (OAuth 불필요)\n- MoviePy로 무료 영상 편집\n- AI 기반 향상\n- 최종 영상 다운로드\n\n---\n\n### ⚙️ 실행 전 필수\n1. **런타임 → 런타임 유형 변경 → GPU (T4)** 설정\n2. Chrome Extension에서 JSON 파일 다운로드 완료

## 1️⃣ 환경 설정 및 라이브러리 설치

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

# 필수 라이브러리 설치
!pip install -q moviepy pillow numpy opencv-python-headless
!pip install -q imageio imageio-ffmpeg
!apt-get update && apt-get install -y ffmpeg fonts-noto-cjk

import os
import json
import base64
from io import BytesIO
from datetime import datetime
import moviepy.editor as mpe
from moviepy.editor import *
import numpy as np
import cv2
from PIL import Image
from IPython.display import Video, display, HTML
from google.colab import files

print("✅ 환경 설정 완료")

## 2️⃣ Chrome Extension JSON 파일 업로드

In [ ]:
print("="*50)
print("📁 Chrome Extension에서 다운로드한 JSON 파일을 업로드하세요")
print("="*50)

# 파일 업로드
uploaded = files.upload()

if not uploaded:
    print("⚠️ 파일이 업로드되지 않았습니다.")
else:
    # JSON 파일 읽기
    json_filename = list(uploaded.keys())[0]
    with open(json_filename, 'r') as f:
        extension_data = json.load(f)
    
    SESSION_ID = extension_data.get('session_id', 'unknown')
    image_count = extension_data.get('image_count', 0)
    
    print(f"\n✅ 파일 로드 성공!")
    print(f"📌 Session ID: {SESSION_ID}")
    print(f"📸 이미지 수: {image_count}개")
    print(f"📅 생성 시간: {extension_data.get('created_at', 'unknown')}")

## 3️⃣ 이미지 처리 및 저장

In [ ]:
# 작업 디렉토리 생성
WORK_DIR = f'/content/session_{SESSION_ID}'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/images', exist_ok=True)
os.makedirs(f'{WORK_DIR}/output', exist_ok=True)

print(f"작업 디렉토리: {WORK_DIR}")
print("\n이미지 처리 중...")

image_files = []
images_data = extension_data.get('images', [])

for idx, img_data in enumerate(images_data):
    try:
        if img_data['url'].startswith('data:'):
            # Data URL 디코딩
            header, encoded = img_data['url'].split(',', 1)
            img_bytes = base64.b64decode(encoded)
            img = Image.open(BytesIO(img_bytes))
        else:
            # URL에서 다운로드 (필요시)
            print(f"  ⚠️ URL 이미지는 수동 다운로드 필요: {img_data['url'][:50]}...")
            continue
        
        # 이미지 저장
        img_filename = img_data.get('filename', f'image_{idx:03d}.png')
        img_path = f"{WORK_DIR}/images/{img_filename}"
        img.save(img_path)
        image_files.append(img_path)
        
        print(f"  ✓ {img_filename} ({img.size[0]}x{img.size[1]})")
    except Exception as e:
        print(f"  ✗ 이미지 {idx} 처리 실패: {str(e)}")

print(f"\n✅ 총 {len(image_files)}개 이미지 준비 완료")

# 첫 번째 이미지 미리보기
if image_files:
    from IPython.display import Image as IPImage
    display(IPImage(image_files[0], width=400))

## 4️⃣ 영상 편집 설정

In [ ]:
# ===== 영상 편집 설정 (자유롭게 수정) =====
VIDEO_CONFIG = {
    # 기본 설정
    "duration_per_image": 3.0,      # 이미지당 표시 시간(초)
    "fps": 30,                       # 프레임 레이트
    "resolution": (1920, 1080),     # HD: (1280,720), FHD: (1920,1080), 4K: (3840,2160)
    
    # 전환 효과
    "transition_type": "crossfade",  # crossfade, slide, zoom, none
    "transition_duration": 0.5,      # 전환 시간(초)
    
    # 모션 효과
    "enable_ken_burns": True,        # Ken Burns 효과 (줌/패닝)
    "zoom_ratio": 1.2,               # 줌 비율 (1.0 = 줌 없음)
    
    # 색상 필터
    "color_filter": "cinematic",     # none, cinematic, vintage, cold, warm
    "brightness": 1.0,               # 밝기 (0.5 ~ 2.0)
    "contrast": 1.1,                 # 대비 (0.5 ~ 2.0)
    
    # 출력
    "output_format": "mp4",          # mp4, avi, mov, webm
    "quality": "high",               # low, medium, high, ultra
}

# 품질 프리셋
QUALITY_PRESETS = {
    "low": {"bitrate": "2M", "crf": 28},
    "medium": {"bitrate": "5M", "crf": 23},
    "high": {"bitrate": "10M", "crf": 18},
    "ultra": {"bitrate": "20M", "crf": 15}
}

total_duration = len(image_files) * VIDEO_CONFIG['duration_per_image']
print(f"⚙️ 영상 편집 설정 완료")
print(f"📹 예상 영상 길이: {total_duration:.1f}초 ({total_duration/60:.1f}분)")
print(f"🎬 해상도: {VIDEO_CONFIG['resolution'][0]}x{VIDEO_CONFIG['resolution'][1]}")
print(f"🎨 필터: {VIDEO_CONFIG['color_filter']}")

## 5️⃣ 영상 편집 함수

In [ ]:
def create_ken_burns_effect(image_path, duration, zoom_ratio=1.2):
    """Ken Burns 효과 생성"""
    clip = ImageClip(image_path, duration=duration)
    w, h = clip.size
    
    # 줌 인/아웃 애니메이션
    def make_frame(t):
        zoom = 1 + (zoom_ratio - 1) * (t / duration)
        new_w, new_h = int(w * zoom), int(h * zoom)
        
        # 중앙 크롭
        x = (new_w - w) // 2
        y = (new_h - h) // 2
        
        return clip.resize((new_w, new_h)).crop(x1=x, y1=y, x2=x+w, y2=y+h).get_frame(t)
    
    return VideoClip(make_frame, duration=duration).set_fps(30)

def apply_color_filter(clip, filter_type="cinematic"):
    """색상 필터 적용"""
    if filter_type == "cinematic":
        # 시네마틱: 대비 증가, 채도 약간 감소
        return clip.fx(vfx.colorx, 0.95).fx(vfx.lum_contrast, contrast=0.2)
    elif filter_type == "vintage":
        # 빈티지: 따뜻한 톤, 대비 감소
        return clip.fx(vfx.colorx, 1.1).fx(vfx.lum_contrast, contrast=-0.1)
    elif filter_type == "cold":
        # 차가운 톤
        return clip.fx(vfx.colorx, 0.9)
    elif filter_type == "warm":
        # 따뜻한 톤
        return clip.fx(vfx.colorx, 1.15)
    return clip

print("✅ 편집 함수 준비 완료")

## 6️⃣ 메인 영상 생성

In [ ]:
print("🎬 영상 생성 시작...\n")

clips = []
for idx, img_path in enumerate(image_files):
    print(f"처리 중: {idx+1}/{len(image_files)} - {os.path.basename(img_path)}")
    
    # 기본 클립 생성
    duration = VIDEO_CONFIG['duration_per_image']
    
    # Ken Burns 효과 적용
    if VIDEO_CONFIG['enable_ken_burns']:
        try:
            clip = create_ken_burns_effect(img_path, duration, VIDEO_CONFIG['zoom_ratio'])
        except:
            # 에러 시 기본 클립 사용
            clip = ImageClip(img_path, duration=duration)
    else:
        clip = ImageClip(img_path, duration=duration)
    
    # 해상도 조정
    clip = clip.resize(VIDEO_CONFIG['resolution'])
    
    # 색상 필터 적용
    if VIDEO_CONFIG['color_filter'] != 'none':
        clip = apply_color_filter(clip, VIDEO_CONFIG['color_filter'])
    
    # 밝기 조정
    if VIDEO_CONFIG['brightness'] != 1.0:
        clip = clip.fx(vfx.colorx, VIDEO_CONFIG['brightness'])
    
    clips.append(clip)

# 클립 연결
print("\n🔗 클립 연결 중...")

if VIDEO_CONFIG['transition_type'] == 'crossfade' and len(clips) > 1:
    # 크로스페이드 전환
    trans_duration = VIDEO_CONFIG['transition_duration']
    final_clips = [clips[0]]
    
    for i in range(1, len(clips)):
        clips[i] = clips[i].crossfadein(trans_duration)
        clips[i] = clips[i].set_start(final_clips[-1].duration - trans_duration)
        final_clips.append(clips[i])
    
    video = CompositeVideoClip(final_clips)
else:
    # 단순 연결
    video = concatenate_videoclips(clips, method="compose")

print(f"\n✅ 영상 생성 완료: {video.duration:.1f}초")

## 7️⃣ BGM 추가 (선택사항)

In [ ]:
# 간단한 BGM 생성 (선택사항)
ADD_BGM = True  # BGM 추가 여부

if ADD_BGM:
    print("🎵 BGM 생성 중...")
    
    # 사인파 기반 앰비언트 사운드 생성
    from moviepy.audio.AudioClip import AudioClip
    
    def make_ambient_sound(t):
        """앰비언트 사운드 생성"""
        # 낮은 주파수 조합으로 부드러운 배경음 생성
        frequencies = [110, 165, 220]  # A2, E3, A3
        signal = np.zeros_like(t)
        for freq in frequencies:
            signal += np.sin(2 * np.pi * freq * t) * 0.05
        return np.stack([signal, signal]).T if len(signal.shape) == 1 else signal
    
    audio = AudioClip(make_ambient_sound, duration=video.duration)
    audio = audio.volumex(0.3)  # 볼륨 조절
    audio = audio.audio_fadein(2).audio_fadeout(2)  # 페이드 인/아웃
    
    video = video.set_audio(audio)
    print("✅ BGM 추가 완료")
else:
    print("⏭️ BGM 추가 건너뜀")

## 8️⃣ 최종 렌더링

In [ ]:
# 출력 파일 설정
output_filename = f"video_{SESSION_ID}.{VIDEO_CONFIG['output_format']}"
output_path = f"{WORK_DIR}/output/{output_filename}"

# 품질 설정
quality = QUALITY_PRESETS[VIDEO_CONFIG['quality']]

print(f"🎬 최종 렌더링 시작...")
print(f"📁 출력: {output_path}")
print(f"🎯 품질: {VIDEO_CONFIG['quality']} (bitrate: {quality['bitrate']}, crf: {quality['crf']})")
print(f"⏱️ 예상 시간: 1-3분 (GPU 사용 시 더 빠름)\n")

# 렌더링 실행
video.write_videofile(
    output_path,
    fps=VIDEO_CONFIG['fps'],
    codec='libx264',
    bitrate=quality['bitrate'],
    audio_codec='aac' if ADD_BGM else None,
    audio_bitrate='128k' if ADD_BGM else None,
    preset='medium',  # ultrafast, fast, medium, slow
    threads=4
)

# 파일 크기 확인
file_size = os.path.getsize(output_path) / (1024*1024)
print(f"\n✅ 렌더링 완료!")
print(f"📁 파일 크기: {file_size:.1f} MB")
print(f"⏱️ 영상 길이: {video.duration:.1f}초")

## 9️⃣ 미리보기 및 다운로드

In [ ]:
# 영상 미리보기
print("🎥 영상 미리보기:")
display(Video(output_path, embed=True, width=720))

print("\n" + "="*50)
print("💾 영상 다운로드:")
print("아래 버튼을 클릭하여 영상을 다운로드하세요.")
files.download(output_path)

## 🔟 추가 편집 옵션 (선택사항)

In [ ]:
# 텍스트 오버레이 추가하기
def add_text_to_video(video, text, position='bottom', fontsize=50):
    """비디오에 텍스트 추가"""
    txt_clip = TextClip(
        text,
        fontsize=fontsize,
        color='white',
        stroke_color='black',
        stroke_width=2,
        font='Arial'
    )
    txt_clip = txt_clip.set_position(position).set_duration(video.duration)
    return CompositeVideoClip([video, txt_clip])

# 워터마크 추가하기 (예시)
# video_with_text = add_text_to_video(video, "Made with Chrome Extension", position='bottom')

# 속도 조절 (2배속)
# fast_video = video.speedx(2)

# 역재생
# reverse_video = video.fx(vfx.time_mirror)

print("🎨 추가 편집 옵션들:")
print("- add_text_to_video(): 텍스트 오버레이")
print("- video.speedx(2): 속도 조절")
print("- video.fx(vfx.time_mirror): 역재생")
print("- video.rotate(90): 회전")
print("- video.resize(0.5): 크기 조절")

---\n\n## 📚 사용 완료!\n\n### 다음 단계:\n1. **영상 다운로드**: 위 셀에서 다운로드 버튼 클릭\n2. **YouTube 업로드**: 다운로드한 영상을 YouTube에 업로드\n3. **공유**: SNS나 메신저로 공유\n\n### 문제 해결:\n- **메모리 부족**: 이미지 수를 줄이거나 해상도를 낮추세요\n- **렌더링 실패**: GPU 런타임을 사용하고 있는지 확인\n- **품질 문제**: VIDEO_CONFIG에서 'quality'를 'ultra'로 설정\n\n---\n\n*Chrome Extension + Colab 영상 편집 파이프라인 v2.0*\n*OAuth 없이 간편하게 사용 가능*